In [ ]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.8 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

# Paths
FEVER_PATH   = Path("train.jsonl")          # FEVER queries
KILT_PATHS   = sorted(Path(".").glob("kilt_passages_*.jsonl"))  # one or more KILT shards
OUTPUT_PATH  = Path("fever_retrieved_docs.jsonl")

# Model
MODEL_NAME   = "all-MiniLM-L6-v2"
# Retrieval
TOP_K_PASSAGES = 100   # passages fetched from FAISS per query (pre-aggregation)
TOP_N_DOCS     = 5     # final doc_ids kept per query (post-aggregation)

ENCODE_BATCH   = 512   # sentences per encoding batch
USE_GPU        = True # set True if a CUDA GPU is available

print(f"FEVER file : {FEVER_PATH}  (exists={FEVER_PATH.exists()})")
print(f"KILT shards: {[p.name for p in KILT_PATHS]}")
print(f"Output     : {OUTPUT_PATH}")

FEVER file : train.jsonl  (exists=True)
KILT shards: ['kilt_passages_00000.jsonl', 'kilt_passages_00001.jsonl']
Output     : fever_retrieved_docs.jsonl


## Load KILT passages

In [ ]:
import json

passage_ids   = []   # "324_1", "324_2", ...
doc_ids       = []   # "324", "324", ...
titles        = []   # "Academy Awards", ...
texts         = []   # passage text

for kilt_path in KILT_PATHS:
    print(f"Loading {kilt_path.name} …")
    with open(kilt_path) as f:
        for line in f:
            row = json.loads(line)
            passage_ids.append(row["passage_id"])
            doc_ids.append(row["doc_id"])
            titles.append(row["title"])
            # Prepend title to give the encoder useful context
            texts.append(f"{row['title']} {row['text']}")

print(f"\nTotal passages : {len(texts):,}")
print(f"Unique doc_ids : {len(set(doc_ids)):,}")
print(f"\nSample passage:")
print(f"  passage_id : {passage_ids[0]}")
print(f"  doc_id     : {doc_ids[0]}")
print(f"  text[:120] : {texts[0][:120]}")

Loading kilt_passages_00000.jsonl …
Loading kilt_passages_00001.jsonl …

Total passages : 663,915
Unique doc_ids : 13,868

Sample passage:
  passage_id : 324_1
  doc_id     : 324
  text[:120] : Academy Awards The Academy Awards, also officially and popularly known as the Oscars, are awards for artistic and techni


## Load FEVER claims

In [ ]:
fever_rows = []
claims     = []

with open(FEVER_PATH) as f:
    for line in f:
        row = json.loads(line)
        fever_rows.append(row)
        claims.append(row["claim"])

print(f"Total claims: {len(claims):,}")
print(f"\nSample claims:")
for c in claims[:5]:
    print(f"  {c}")

Total claims: 145,449

Sample claims:
  Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.
  Roman Atwood is a content creator.
  History of art includes architecture, dance, sculpture, music, painting, poetry literature, theatre, narrative, film, photography and graphic arts.
  Adrienne Bailon is an accountant.
  System of a Down briefly disbanded in limbo.


## Embed passages & FAISS index

Passages are embedded once and stored in a FAISS `IndexFlatIP` (exact inner-product search).  
Because vectors are L2-normalised, inner product equals cosine similarity.

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print(f"Loading model '{MODEL_NAME}'")
model = SentenceTransformer(MODEL_NAME)
device = "cuda" if USE_GPU else "cpu"
model = model.to(device)

Loading model 'all-MiniLM-L6-v2' …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Model loaded on cuda


In [ ]:
print(f"Encoding {len(texts):,} passages (batch={ENCODE_BATCH}) …")

passage_embs = model.encode(
    texts,
    batch_size=ENCODE_BATCH,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2-norm → cosine sim = dot product
    convert_to_numpy=True,
    device=device,
).astype("float32")

print(f"Passage embedding matrix: {passage_embs.shape}")

Encoding 663,915 passages (batch=512) …


Batches:   0%|          | 0/1297 [00:00<?, ?it/s]

Passage embedding matrix: (663915, 384)


In [ ]:
dim   = passage_embs.shape[1]
index = faiss.IndexFlatIP(dim)

if USE_GPU:
    try:
        res   = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    except AttributeError:
        print("faiss-gpu not installed, falling back to CPU index")

index.add(passage_embs)
print(f"FAISS index built — {index.ntotal:,} vectors, dim={dim}")

faiss-gpu not installed, falling back to CPU index
FAISS index built — 663,915 vectors, dim=384


## Embed claims and search

In [ ]:
print(f"Encoding {len(claims):,} claims …")

claim_embs = model.encode(
    claims,
    batch_size=ENCODE_BATCH,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
    device=device,
).astype("float32")

print(f"Claim embedding matrix: {claim_embs.shape}")

Encoding 145,449 claims …


Batches:   0%|          | 0/285 [00:00<?, ?it/s]

Claim embedding matrix: (145449, 384)


In [ ]:
print(f"Searching top-{TOP_K_PASSAGES} passages per claim …")

# scores: (n_claims, TOP_K_PASSAGES)  — cosine similarities
# indices: (n_claims, TOP_K_PASSAGES) — passage positions in index
scores, indices = index.search(claim_embs, TOP_K_PASSAGES)

print(f"Search complete. Result shape: {scores.shape}")

Searching top-100 passages per claim …
Search complete. Result shape: (145449, 100)


In [ ]:
doc_ids_arr = np.array(doc_ids)  # index-aligned with FAISS

retrieved = []  # one dict per claim

for i in range(len(claims)):
    hit_indices = indices[i]   # passage positions
    hit_scores  = scores[i]    # cosine similarities

    # Aggregate: max score per doc_id
    doc_score: dict[str, float] = {}
    doc_best_passage: dict[str, str] = {}

    for idx, score in zip(hit_indices, hit_scores):
        did  = doc_ids_arr[idx]
        pid  = passage_ids[idx]
        if did not in doc_score or score > doc_score[did]:
            doc_score[did]        = float(score)
            doc_best_passage[did] = pid

    # Sort by score descending, keep top-N
    ranked_docs = sorted(doc_score, key=doc_score.__getitem__, reverse=True)[:TOP_N_DOCS]

    retrieved.append({
        "doc_ids"       : ranked_docs,
        "doc_scores"    : [round(doc_score[d], 4)        for d in ranked_docs],
        "best_passages" : [doc_best_passage[d]           for d in ranked_docs],
        "best_titles"   : [titles[passage_ids.index(doc_best_passage[d])] for d in ranked_docs],
    })

print(f"Aggregation done for {len(retrieved):,} claims")

Aggregation done for 145,449 claims


### Results

In [ ]:
import pandas as pd

for i in range(5):
    row = fever_rows[i]
    ret = retrieved[i]
    print(f"Claim [{row['id']}]: {row['claim']}")
    for j, (did, score, pid, title) in enumerate(
        zip(ret["doc_ids"], ret["doc_scores"], ret["best_passages"], ret["best_titles"])
    ):
        print(f"  #{j+1}  doc_id={did:>12}  score={score:.4f}  title='{title}'  best_passage={pid}")
    print()

Claim [75397]: Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.
  #1  doc_id=     2316131  score=0.6115  title='Nikolaj Coster-Waldau'  best_passage=2316131_25
  #2  doc_id=     1200093  score=0.5423  title='Larry the Cable Guy'  best_passage=1200093_29
  #3  doc_id=    48493547  score=0.5074  title='Sacha Pfeiffer'  best_passage=48493547_10
  #4  doc_id=      170318  score=0.5047  title='20th Century Fox'  best_passage=170318_5
  #5  doc_id=     1686616  score=0.4930  title='Kerry Fox'  best_passage=1686616_7

Claim [150448]: Roman Atwood is a content creator.
  #1  doc_id=    43386202  score=0.6480  title='Roman Atwood'  best_passage=43386202_3
  #2  doc_id=        1273  score=0.4920  title='Augustus'  best_passage=1273_212
  #3  doc_id=       21632  score=0.4843  title='Nero'  best_passage=21632_149
  #4  doc_id=       59375  score=0.4751  title='Titus Andronicus'  best_passage=59375_276
  #5  doc_id=       60280  score=0.4724  title='Margaret Atwood'  best_passage=60

## Evaluate recall against evidence


In [ ]:
def normalise(t):
    return t.replace("_", " ").replace("-LRB-", "(").replace("-RRB-", ")").lower().strip()

def extract_gold_titles(evidence_field):
    titles_found = set()
    for evidence_set in evidence_field:
        for ev in evidence_set:
            if len(ev) >= 3 and ev[2] is not None:
                titles_found.add(ev[2])
    return titles_found

# Build normalised title → doc_id lookup
title_to_docid = {}
for did, title in zip(doc_ids, titles):
    norm = normalise(title)
    if norm not in title_to_docid:
        title_to_docid[norm] = did

# Compute Recall@K for K in {1, 2, 3, 5}
hits = {k: 0 for k in [1, 2, 3, 5]}
total_verifiable = 0

for row, ret in zip(fever_rows, retrieved):
    gold_titles = extract_gold_titles(row.get("evidence", []))
    if not gold_titles:
        continue
    total_verifiable += 1

    # Normalise gold titles before lookup
    gold_docids = {title_to_docid.get(normalise(t)) for t in gold_titles} - {None}
    retrieved_docids = ret["doc_ids"]

    for k in hits:
        if gold_docids & set(retrieved_docids[:k]):
            hits[k] += 1

print(f"Evaluated on {total_verifiable:,} claims with evidence annotations")
for k, h in hits.items():
    print(f"  Recall@{k} = {h/total_verifiable:.4f}  ({h}/{total_verifiable})")

Evaluated on 109,810 claims with evidence annotations

  Recall@1 = 0.7732  (84903/109810)
  Recall@2 = 0.8987  (98682/109810)
  Recall@3 = 0.9302  (102145/109810)
  Recall@5 = 0.9475  (104041/109810)


## Save output

In [ ]:
with open(OUTPUT_PATH, "w") as f:
    for row, ret in zip(fever_rows, retrieved):
        out = dict(row)
        out["top_doc_ids"] = ret["doc_ids"][:5]
        f.write(json.dumps(out) + "\n")

print(f"Saved {len(fever_rows):,} rows → {OUTPUT_PATH}")

with open(OUTPUT_PATH) as f:
    sample = json.loads(f.readline())
print("\nSample output row:")
print(json.dumps(sample, indent=2))

Saved 145,449 rows → fever_retrieved_docs.jsonl

Sample output row:
{
  "id": 75397,
  "verifiable": "VERIFIABLE",
  "label": "SUPPORTS",
  "claim": "Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.",
  "evidence": [
    [
      [
        92206,
        104971,
        "Nikolaj_Coster-Waldau",
        7
      ],
      [
        92206,
        104971,
        "Fox_Broadcasting_Company",
        0
      ]
    ]
  ],
  "top_doc_ids": [
    "2316131",
    "1200093",
    "48493547",
    "170318",
    "1686616"
  ]
}


In [ ]:
# not used - outputing just the doc ids
# with open(OUTPUT_PATH, "w") as f:
#     for row, ret in zip(fever_rows, retrieved):
#         out = dict(row)
#         out["retrieved"] = [
#             {
#                 "doc_id"          : did,
#                 "score"           : score,
#                 "best_passage_id" : pid,
#                 "title"           : title,
#             }
#             for did, score, pid, title in zip(
#                 ret["doc_ids"],
#                 ret["doc_scores"],
#                 ret["best_passages"],
#                 ret["best_titles"],
#             )
#         ]
#         f.write(json.dumps(out) + "\n")

# print(f"Saved {len(fever_rows):,} rows → {OUTPUT_PATH}")

# with open(OUTPUT_PATH) as f:
#     sample = json.loads(f.readline())
# print("\nSample output row:")
# print(json.dumps(sample, indent=2))

Saved 145,449 rows → fever_retrieved_docs.jsonl

Sample output row:
{
  "id": 75397,
  "verifiable": "VERIFIABLE",
  "label": "SUPPORTS",
  "claim": "Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.",
  "evidence": [
    [
      [
        92206,
        104971,
        "Nikolaj_Coster-Waldau",
        7
      ],
      [
        92206,
        104971,
        "Fox_Broadcasting_Company",
        0
      ]
    ]
  ],
  "retrieved": [
    {
      "doc_id": "2316131",
      "score": 0.6115,
      "best_passage_id": "2316131_25",
      "title": "Nikolaj Coster-Waldau"
    },
    {
      "doc_id": "1200093",
      "score": 0.5423,
      "best_passage_id": "1200093_29",
      "title": "Larry the Cable Guy"
    },
    {
      "doc_id": "48493547",
      "score": 0.5074,
      "best_passage_id": "48493547_10",
      "title": "Sacha Pfeiffer"
    },
    {
      "doc_id": "170318",
      "score": 0.5047,
      "best_passage_id": "170318_5",
      "title": "20th Century Fox"
    },
 